# Maia2 personal fine-tune for `nick_p12`

Fine-tunes the Maia2 base **blitz** model on your own PGN games so it
plays your openings and roughly your strength.

**Repo:** [`nikhileshp/chess-clone`](https://github.com/nikhileshp/chess-clone)

**Before running:** Runtime → Change runtime type → GPU → A100 (or L4 / V100).

Two strategies in sequence:
* **Plan A** — use the package's built-in `maia2.train.run(cfg)` with a
  fine-tuning YAML (low LR, load pretrained, few epochs). Cleanest if
  the package exposes the right knobs.
* **Plan B** — if Plan A fails, fall back to a hand-written PyTorch loop
  on top of `model.from_pretrained()`.

## 1. Environment + GPU check

In [ ]:
!nvidia-smi

## 2. Clone the repo (code + compressed game data live here)

In [ ]:
%%bash
set -e
rm -rf /content/repo
git clone --depth 1 https://github.com/nikhileshp/chess-clone.git /content/repo
ls -lh /content/repo/data/clean/ || true
echo '---'
ls /content/repo/colab/

In [ ]:
import sys
sys.path.insert(0, '/content/repo')
sys.path.insert(0, '/content/repo/colab')
from finetune_helpers import FineTuneConfig, set_seed, log_environment, write_run_metadata, write_yaml_config
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
set_seed(42)
ENV = log_environment()

## 3. Install Maia2

PyTorch 2.4 + a few helpers. The package ships its own data loader that
reads `.pgn.zst` directly.

In [ ]:
# Note: maia2's setup.py is incomplete — it omits both python-chess and pyzstd
# as dependencies. We install them explicitly here. Without this, imports fail
# with `ModuleNotFoundError: No module named 'chess'` or `'pyzstd'`.
!pip install -q maia2 chess pyzstd zstandard pyyaml
!python -c 'import maia2, chess, pyzstd, torch; print("maia2:", getattr(maia2, "__version__", "?")); print("chess:", chess.__version__); print("pyzstd:", pyzstd.__version__); print("torch:", torch.__version__); print("cuda:", torch.cuda.is_available())'

## 4. Inspect the package surface

The Maia2 README gives a high-level outline but exact training/config
knobs aren't documented. We introspect the installed package so the
rest of this notebook can adapt to whatever the actual API is.

In [ ]:
import inspect
from maia2 import model, train, utils
from maia2 import inference, dataset

print('=== maia2.model exports ===')
print([n for n in dir(model) if not n.startswith('_')])
print()
print('=== maia2.train exports ===')
print([n for n in dir(train) if not n.startswith('_')])
print()
print('=== maia2.dataset exports ===')
print([n for n in dir(dataset) if not n.startswith('_')])
print()
print('=== model.from_pretrained signature ===')
print(inspect.signature(model.from_pretrained))
print()
if hasattr(train, 'run'):
    print('=== train.run signature ===')
    print(inspect.signature(train.run))
    print(inspect.getsource(train.run)[:1500])

## 5. Configure the fine-tune

In [ ]:
cfg = FineTuneConfig(
    user='nick_p12',
    pgn_zst_path='/content/repo/data/clean/all.pgn.zst',
    output_dir='/content/repo/colab_outputs',
    pretrained_type='blitz',
    epochs=3,
    learning_rate=1e-5,
    batch_size=256,
    val_fraction=0.05,
    freeze_encoder=False,
    weight_decay=1e-4,
    warmup_steps=200,
)
from pathlib import Path
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
meta_path = write_run_metadata(cfg, ENV, Path(cfg.output_dir))
yaml_path = write_yaml_config(cfg, Path(cfg.output_dir) / 'finetune.yaml')
print(open(yaml_path).read())

## 6. Plan A — try `maia2.train.run(cfg)` with our YAML

If this cell errors with a schema mismatch, read the error message and
either adjust `finetune_helpers.write_yaml_config` to match the package's
expected schema, OR skip to Plan B below.

In [ ]:
PLAN_A_OK = False
try:
    if hasattr(train, 'run'):
        train.run(str(yaml_path))
        PLAN_A_OK = True
    else:
        print('train.run() not available — falling through to Plan B')
except Exception as e:
    print(f'Plan A failed: {type(e).__name__}: {e}')
    print('Continue to Plan B (custom loop) below.')
print('Plan A succeeded:', PLAN_A_OK)

## 7. Plan B — custom PyTorch fine-tune loop

Only run these cells if Plan A failed. Loads the pretrained model
directly, builds a Dataset/DataLoader from your `.pgn.zst`, and trains a
few epochs of cross-entropy on your actual moves.

We import any data-prep helpers Maia2 exposes (e.g. `dataset.PGNDataset`)
rather than reinventing the position encoding.

In [ ]:
if not PLAN_A_OK:
    import torch
    from torch.utils.data import DataLoader
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('device:', device)
    
    # Load pretrained
    m = model.from_pretrained(type=cfg.pretrained_type, device='gpu' if device.type == 'cuda' else 'cpu')
    print('model class:', type(m).__name__)
    print('model param count:', sum(p.numel() for p in m.parameters()))
    print('model.forward signature:', inspect.signature(m.forward) if hasattr(m, 'forward') else 'n/a')
    
    # Inspect the dataset module to find a PGN-ingest class
    print('=== maia2.dataset attrs ===')
    for name in dir(dataset):
        if name.startswith('_'): continue
        obj = getattr(dataset, name)
        kind = type(obj).__name__
        print(f'  {name}: {kind}')

In [ ]:
# =====================================================================
# Plan B — real implementation (uses maia2's own board_to_tensor +
# all_moves_dict so the fine-tune is consistent with pretraining).
# Run this cell once. ~5-10 min to build dataset, ~1-2 hr to train on T4.
# =====================================================================
import io, time, random, gzip
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import chess, chess.pgn
import pyzstd
from tqdm.auto import tqdm

from maia2.utils import board_to_tensor, get_all_possible_moves

# 1) Move-index dictionary (must match pretraining)
ALL_MOVES_DICT = {m: i for i, m in enumerate(get_all_possible_moves())}
N_MOVES = len(ALL_MOVES_DICT)
print(f'Action space: {N_MOVES} moves')

# 2) Stream-parse our .pgn.zst into (fen, move_uci, elo_self, elo_oppo, active_win) tuples.
#    'active_win' = 1 if the side-to-move won that game, 0 if drew, -1 if lost.
USER = cfg.user.lower()

def parse_pgn_zst(path: str, max_games: int | None = None):
    samples = []
    with pyzstd.open(path, 'rb') as fz:
        text = fz.read().decode('utf-8', errors='replace')
    buf = io.StringIO(text)
    n_games = 0
    while True:
        game = chess.pgn.read_game(buf)
        if game is None:
            break
        n_games += 1
        if max_games and n_games > max_games:
            break
        h = game.headers
        white = (h.get('White') or '').lower()
        black = (h.get('Black') or '').lower()
        user_is_white = (white == USER)
        if not user_is_white and black != USER:
            continue
        try:
            elo_self = int(h.get('WhiteElo' if user_is_white else 'BlackElo', '?'))
            elo_oppo = int(h.get('BlackElo' if user_is_white else 'WhiteElo', '?'))
        except ValueError:
            continue
        result = h.get('Result', '*')
        if result == '1-0':
            user_score = 1 if user_is_white else 0
        elif result == '0-1':
            user_score = 0 if user_is_white else 1
        elif result == '1/2-1/2':
            user_score = 0.5
        else:
            continue

        board = game.board()
        node = game
        while node.variations:
            node = node.variation(0)
            move = node.move
            user_to_move = (board.turn == chess.WHITE) == user_is_white
            if user_to_move:
                uci = move.uci()
                if uci in ALL_MOVES_DICT:  # skip moves outside action space (rare)
                    # active_win: from the side-to-move's perspective
                    aw = 1 if user_score == 1 else (0 if user_score == 0.5 else -1)
                    samples.append((board.fen(), uci, elo_self, elo_oppo, aw))
            board.push(move)
    return samples, n_games

print('Parsing PGN...')
t0 = time.time()
samples, n_games = parse_pgn_zst(cfg.pgn_zst_path)
print(f'  {n_games} games | {len(samples)} user-move samples | {time.time()-t0:.1f}s')

# 3) PyTorch Dataset
class UserMoveDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        fen, uci, elo_s, elo_o, aw = self.rows[i]
        board = chess.Board(fen)
        x = board_to_tensor(board)
        y = ALL_MOVES_DICT[uci]
        return x, y, elo_s, elo_o, aw

random.Random(cfg.seed).shuffle(samples)
n_val = int(len(samples) * cfg.val_fraction)
train_rows = samples[n_val:]
val_rows = samples[:n_val]
print(f'  train={len(train_rows)} val={len(val_rows)}')
train_dl = DataLoader(UserMoveDataset(train_rows), batch_size=cfg.batch_size,
                      shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(UserMoveDataset(val_rows), batch_size=cfg.batch_size,
                    shuffle=False, num_workers=2, pin_memory=True)

# 4) Fine-tune loop
m.train()
opt = torch.optim.AdamW(m.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=cfg.warmup_steps)

def run_eval(loader, label):
    m.eval()
    correct1, correct3, total = 0, 0, 0
    with torch.no_grad():
        for x, y, es, eo, _ in loader:
            x = x.to(device); y = y.to(device)
            es = es.to(device); eo = eo.to(device)
            out = m(x, es, eo)
            logits = out[0] if isinstance(out, (tuple, list)) else out
            top3 = logits.topk(3, dim=-1).indices
            correct1 += (top3[:, 0] == y).sum().item()
            correct3 += (top3 == y.unsqueeze(1)).any(dim=1).sum().item()
            total += y.size(0)
    print(f'  {label}: top-1 {correct1/total:.4f} | top-3 {correct3/total:.4f} | n={total}')
    m.train()
    return correct1 / total

# Baseline before fine-tuning (sanity check)
print('Baseline (pretrained Maia2-blitz, no fine-tune):')
base_top1 = run_eval(val_dl, 'val')

best_val = 0.0
metrics = {'epochs': []}
for epoch in range(cfg.epochs):
    print(f'\\n=== Epoch {epoch+1}/{cfg.epochs} ===')
    t0 = time.time()
    running = 0.0; steps = 0
    for x, y, es, eo, _ in tqdm(train_dl, desc=f'epoch {epoch+1}'):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        es = es.to(device, non_blocking=True)
        eo = eo.to(device, non_blocking=True)
        out = m(x, es, eo)
        logits = out[0] if isinstance(out, (tuple, list)) else out
        loss = F.cross_entropy(logits, y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        sched.step()
        running += loss.item(); steps += 1
    train_loss = running / max(steps, 1)
    val_top1 = run_eval(val_dl, 'val')
    metrics['epochs'].append({
        'epoch': epoch + 1, 'train_loss': train_loss,
        'val_top1': val_top1, 'seconds': time.time() - t0,
    })
    if val_top1 > best_val:
        best_val = val_top1
        ts = int(time.time())
        ckpt = f'{cfg.output_dir}/maia2_finetuned_best.pt'
        torch.save({'state_dict': m.state_dict(),
                    'val_top1': val_top1,
                    'baseline_top1': base_top1,
                    'config': {k: v for k, v in vars(cfg).items()},
                    'epoch': epoch + 1}, ckpt)
        print(f'  saved best -> {ckpt}')

# Final save
ts = int(time.time())
final_path = f'{cfg.output_dir}/maia2_finetuned_{ts}.pt'
torch.save({'state_dict': m.state_dict(),
            'baseline_top1': base_top1,
            'best_val_top1': best_val,
            'config': {k: v for k, v in vars(cfg).items()}}, final_path)
print(f'\\nFinal checkpoint: {final_path}')
print(f'Baseline val top-1: {base_top1:.4f}')
print(f'Best fine-tuned val top-1: {best_val:.4f}')
print(f'Improvement: +{(best_val - base_top1)*100:.2f} pp')

import json
with open(f'{cfg.output_dir}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

## 8. Sanity check: does the fine-tuned model predict your moves?

Pick 50 random positions from your held-out games, ask the model for its
top-3 moves, and report top-1 / top-3 accuracy against the move you
actually played. Should be meaningfully higher than the base Maia2 blitz
model (the difference is the personalization signal).

In [ ]:
# Filled in once Plan A or Plan B has produced a checkpoint.
# Compares: base maia2-blitz vs fine-tuned, top-1 and top-3 accuracy on
# 50 random positions from a held-out subset of your games.
pass

## 9. Pull outputs back to your laptop

Either download via the file widget, or push to a `weights` branch of
the repo. The checkpoint is the only artifact you actually need.

In [ ]:
from google.colab import files
import glob
for p in sorted(glob.glob(f'{cfg.output_dir}/*.pt')):
    print('downloading', p)
    files.download(p)
for p in sorted(glob.glob(f'{cfg.output_dir}/*.json')):
    print('downloading', p)
    files.download(p)